# Fase B — análise e gráficos

Lê os `.jsonl` de `results/` **pelo mesmo carregador do consolidador**
(`evaluation.consolidate_phase_b.carrega`), de propósito: o filtro de
procedência — o que torna uma execução válida ou descartável — fica num
lugar só. A versão anterior deste notebook relia os arquivos por conta
própria, sem filtro nenhum, e portanto plotava execuções com rádio
degradado e thread vazada junto das boas.

`carrega` também agrega na leitura em vez de materializar os eventos:
223 execuções × ~26 mil eventos são ~5,8 milhões de dicts, o que já
custou uma consolidação de 3 horas quando foi feito do jeito ingênuo.

Rode as células em ordem.


In [ ]:
import sys
from collections import Counter, defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# o notebook vive em evaluation/; a raiz do repo precisa estar no path
# para `import evaluation.*` funcionar seja qual for o cwd do Jupyter
_aqui = Path.cwd()
_raiz = next((p for p in (_aqui, *_aqui.parents) if (p / "evaluation").is_dir()), None)
if _raiz is None:
    raise SystemExit("rode o notebook de dentro do repositorio")
sys.path.insert(0, str(_raiz))

from evaluation.consolidate_phase_b import (  # noqa: E402
    CENARIOS, MARK_POR_FATIA, RESULTS_DIR, SEM_ORQUESTRADOR, carrega, pct,
)

# Janela da campanha v2 (bloco de 240 execucoes iniciado em 2026-10-21
# 18:07). `results/` acumula blocos de campanhas anteriores com os mesmos
# prefixos de cenario, entao SEM esta janela os graficos misturariam
# campanhas com codigo diferente. E o mesmo valor passado ao consolidador
# em `--desde`, e e por isso que os totais daqui batem com
# docs/FASE-B-DADOS-CONSOLIDADOS.md.
DESDE = 1792606020.0
ATE = None

## Paleta

Slots 1–6 da paleta de referência validada (`skill dataviz →
references/palette.md`), **na ordem documentada** — essa ordem é o
mecanismo de segurança para daltonismo, não enfeite, e está registrada
como aprovada em todos os gates para linhas e barras (pior par adjacente
ΔE CVD 9,1 no claro). A VM não tem `node`, então o validador não pôde
ser rodado aqui; usar o prefixo da ordem publicada é o caminho correto
justamente por isso, em vez de inventar hues.

Três slots do claro (aqua, amarelo, magenta) ficam abaixo de 3:1 de
contraste na superfície clara: vale a regra de alívio, e por isso todo
gráfico aqui sai com legenda **e** a tabela de valores ao lado.

In [ ]:
# ordem fixa; hue segue a entidade, nunca o rank
CENARIO_COR = {
    "cenario0_baseline_nativo": "#2a78d6",   # 1 blue
    "cenario1_pqc_estatico":    "#eb6834",   # 2 orange
    "cenario2_protecao_maxima": "#1baf7a",   # 3 aqua
    "cenario3_risk_aware":      "#eda100",   # 4 yellow
    "cenario4a_escassez_qkd":   "#e87ba4",   # 5 magenta
    "cenario4_contencao":       "#008300",   # 6 green
}
CENARIO_ROTULO = dict((pref, nome) for nome, pref in CENARIOS)
ORDEM = [pref for _, pref in CENARIOS]

# fatias: slots 1-3, que sao os unicos validados para "todos os pares"
FATIA_COR = {"URLLC": "#2a78d6", "EMBB": "#eb6834", "MIOT": "#1baf7a"}
FATIA_ROTULO = {"URLLC": "URLLC", "EMBB": "eMBB", "MIOT": "mIoT"}
ORDEM_FATIA = ["URLLC", "EMBB", "MIOT"]

INK, INK2, GRID, SURFACE = "#0b0b0b", "#52514e", "#e1e0d9", "#fcfcfb"
plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE,
    "axes.edgecolor": GRID, "axes.labelcolor": INK,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.8,
    "axes.axisbelow": True,
    "text.color": INK, "xtick.color": INK2, "ytick.color": INK2,
    "font.size": 11, "axes.spines.top": False, "axes.spines.right": False,
    "legend.frameon": False, "figure.dpi": 120,
})


def barras(ax, grupos, series, valores, cores, rotulos, largura_total=0.78):
    """Barras agrupadas com folga de 2 px entre vizinhas e topo arredondado."""
    x = np.arange(len(grupos))
    w = largura_total / max(len(series), 1)
    for i, s in enumerate(series):
        desloc = -largura_total / 2 + w * (i + 0.5)
        v = [valores[s][g] for g in grupos]
        ax.bar(x + desloc, v, w * 0.88, label=rotulos.get(s, s),
               color=cores[s], linewidth=0)
        # Uma barra de altura zero e indistinguivel de serie ausente, e
        # aqui o zero E o resultado (a URLLC nao rebaixa nunca). Rotula.
        for xi, vi in zip(x, v):
            if not vi:
                ax.annotate("0", (xi + desloc, 0), ha="center", va="bottom",
                            fontsize=8, color=cores[s], fontweight="bold",
                            xytext=(0, 2), textcoords="offset points")
    ax.set_xticks(x)
    ax.set_xticklabels([CENARIO_ROTULO.get(g, g) for g in grupos],
                       rotation=20, ha="right")
    return x

## Carregamento e procedência

A tabela abaixo tem de bater com a seção "Procedência das execuções" de
`docs/FASE-B-DADOS-CONSOLIDADOS.md`. Se não bater, a janela (`DESDE`)
está diferente da que gerou o documento — e aí os gráficos não
correspondem às tabelas do artigo.

In [ ]:
execucoes: dict[str, list[dict]] = defaultdict(list)
procedencia: dict[str, Counter] = defaultdict(Counter)

for _nome, pref in CENARIOS:
    for caminho in sorted(RESULTS_DIR.glob(f"{pref}-*.jsonl")):
        t = caminho.stat().st_mtime
        if DESDE and t < DESDE:
            continue
        if ATE and t > ATE:
            continue
        r = carrega(caminho, pref)
        procedencia[pref][r["condicao"]] += 1
        if r["condicao"] == "valida":
            execucoes[pref].append(r)

if not execucoes:
    raise SystemExit("nenhuma execucao valida na janela — confira DESDE")

presentes = [p for p in ORDEM if execucoes.get(p)]
orquestrados = [p for p in presentes if p not in SEM_ORQUESTRADOR]

tabela_proc = pd.DataFrame(
    [dict(cenario=CENARIO_ROTULO[p], **procedencia[p]) for _, p in CENARIOS
     if procedencia[p]]
).set_index("cenario").fillna(0).astype(int)
tabela_proc["total"] = tabela_proc.sum(axis=1)
display(tabela_proc)
print(f"{tabela_proc['valida'].sum()} validas de {tabela_proc['total'].sum()}")

## Latência URLLC fim a fim — CDF

Fim a fim atravessa o rádio ZMQ, cujo piso é ~55 ms: é a latência que a
fatia realmente vê, mas a cauda é dominada pelo rádio, não pelo IPsec.
O custo atribuível à arquitetura aparece no gráfico seguinte (sonda N3).

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4.6))
linhas = {}
for pref in presentes:
    v = np.sort(np.concatenate([r["latencias"] for r in execucoes[pref]]))
    y = np.arange(1, v.size + 1) / v.size
    ax.plot(v, y, lw=2, color=CENARIO_COR[pref], label=CENARIO_ROTULO[pref])
    linhas[pref] = v

ax.set_xlabel("RTT fim a fim da URLLC (ms)")
ax.set_ylabel("fração das amostras")
ax.set_xlim(left=0)
ax.set_ylim(0, 1.02)
ax.legend(loc="lower right")
ax.set_title("CDF da latência URLLC por cenário", loc="left", pad=12)
plt.tight_layout()
plt.show()

display(pd.DataFrame(
    [{"cenario": CENARIO_ROTULO[p], "n": linhas[p].size,
      "p50": round(float(np.percentile(linhas[p], 50)), 2),
      "p95": round(float(np.percentile(linhas[p], 95)), 2),
      "p99": round(float(np.percentile(linhas[p], 99)), 2)}
     for p in presentes]).set_index("cenario"))

## Latência no túnel N3 (sonda de alta resolução)

Piso de 0,05 ms, sem o rádio no caminho — é aqui que o custo do túnel é
visível. Dois eixos de grandeza diferente (p50 e p99) ficam em dois
painéis, nunca em dois eixos y no mesmo gráfico.

In [ ]:
n3 = {q: {p: _med for p, _med in
          ((p, float(np.median([r["n3"].get(q) for r in execucoes[p]
                                if r["n3"].get(q) is not None] or [np.nan])))
           for p in presentes)} for q in ("p50", "p99")}

fig, axes = plt.subplots(1, 2, figsize=(11, 4.4), sharex=True)
for ax, q, titulo in zip(axes, ("p50", "p99"),
                         ("N3 p50 — mediana entre execuções",
                          "N3 p99 — mediana entre execuções")):
    x = np.arange(len(presentes))
    ax.bar(x, [n3[q][p] for p in presentes], 0.66,
           color=[CENARIO_COR[p] for p in presentes], linewidth=0)
    for xi, p in zip(x, presentes):   # rotulo direto: a cor nao carrega o valor
        ax.annotate(f"{n3[q][p]:.2f}", (xi, n3[q][p]), ha="center",
                    va="bottom", fontsize=9, color=INK2,
                    xytext=(0, 3), textcoords="offset points")
    ax.set_xticks(x)
    ax.set_xticklabels([CENARIO_ROTULO[p] for p in presentes],
                       rotation=25, ha="right")
    ax.set_ylabel("RTT (ms)")
    ax.set_title(titulo, loc="left", pad=10)
plt.tight_layout()
plt.show()

display(pd.DataFrame([{"cenario": CENARIO_ROTULO[p],
                       "N3 p50": round(n3["p50"][p], 3),
                       "N3 p99": round(n3["p99"][p], 3)}
                      for p in presentes]).set_index("cenario"))

## Alocação de entropia por fatia

O gráfico central da proposta. A ordem de prioridade vem do vetor de
criticidade (I > H > D, lexicográfico), que dá **URLLC > mIoT > eMBB** —
a mesma ordem usada pelo escalonador, importada de
`orchestrator/criticality.py` para que as duas não divirjam.

**A contagem crua de `ppk_fetched` não mede a alocação**, e vale
entender por quê antes de ler o gráfico: uma fatia faminta re-tenta a
cada tick (2,5 s), porque a reauth que falha não reautentica nada e
portanto não zera `ike_seconds` — o portão de idade fica aberto até o fim
da execução. Medido: a URLLC tenta 10 vezes em 300 s (a cada 30 s, seu
Tr_PPK, acertando sempre), a mIoT tenta 72 vezes (a cada tick, do
vencimento do prazo em diante). Por acumular tentativas a mIoT acaba
obtendo *mais* chaves que a URLLC em números absolutos, o que inverte a
leitura da tese sem que nada no mecanismo esteja priorizando a mIoT.

A métrica que testa a tese é a **taxa de sucesso**, independente do
número de tentativas — e ela sai na ordem pretendida em todos os
cenários. Em C4a a oferta é escassa de propósito (8,5 bit/s) e só a
URLLC é atendida.

In [ ]:
def por_fatia(chave):
    out = {f: {} for f in ORDEM_FATIA}
    for p in orquestrados:
        for f in ORDEM_FATIA:
            vals = [r["por_fatia"].get(f, {}).get(chave, 0) for r in execucoes[p]]
            out[f][p] = float(np.mean(vals)) if vals else 0.0
    return out


obtido, vazio = por_fatia("ppk_fetched"), por_fatia("ppk_pool_empty")
tentativas = {f: {p: obtido[f][p] + vazio[f][p] for p in orquestrados}
              for f in ORDEM_FATIA}
taxa = {f: {p: (obtido[f][p] / tentativas[f][p] * 100
                if tentativas[f][p] else np.nan) for p in orquestrados}
        for f in ORDEM_FATIA}

fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.6))
barras(axes[0], orquestrados, ORDEM_FATIA, taxa, FATIA_COR, FATIA_ROTULO)
axes[0].set_ylabel("tentativas atendidas (%)")
axes[0].set_ylim(0, 105)
axes[0].set_title("Taxa de sucesso na busca de PPK — a métrica da tese",
                  loc="left", pad=10)
axes[0].legend(ncol=3, loc="lower left", bbox_to_anchor=(0, 1.08))

barras(axes[1], orquestrados, ORDEM_FATIA, tentativas, FATIA_COR, FATIA_ROTULO)
axes[1].set_ylabel("tentativas por execução (média)")
axes[1].set_title("Tentativas — a fatia faminta re-tenta a cada tick",
                  loc="left", pad=10)
plt.tight_layout()
plt.show()

display(pd.DataFrame([{
    "cenario": CENARIO_ROTULO[p], "fatia": FATIA_ROTULO[f],
    "tentativas": round(tentativas[f][p], 1),
    "obtido": round(obtido[f][p], 1),
    "taxa %": round(taxa[f][p], 1) if tentativas[f][p] else None,
} for p in orquestrados for f in ORDEM_FATIA]).set_index(["cenario", "fatia"]))

## Degradação graciosa: downgrades e pool vazio

`rotation_downgraded` é a fatia aceitando um perfil mais fraco por falta
de entropia; `ppk_pool_empty` é a tentativa que não encontrou chave. A
leitura é conjunta: o pool pode esvaziar sem causar downgrade se a
chave chega antes do prazo.

In [ ]:
downgrade = por_fatia("rotation_downgraded")
vazio = por_fatia("ppk_pool_empty")

fig, axes = plt.subplots(1, 2, figsize=(12, 4.4))
for ax, dados, titulo, eixo in (
        (axes[0], downgrade, "Downgrades de perfil", "eventos por execução (média)"),
        (axes[1], vazio, "Tentativas com pool vazio", "eventos por execução (média)")):
    barras(ax, orquestrados, ORDEM_FATIA, dados, FATIA_COR, FATIA_ROTULO)
    ax.set_ylabel(eixo)
    ax.set_title(titulo, loc="left", pad=10)
axes[0].legend(ncol=3, loc="lower left", bbox_to_anchor=(0, 1.08))
plt.tight_layout()
plt.show()

display(pd.concat({
    "downgrades": pd.DataFrame({FATIA_ROTULO[f]: {CENARIO_ROTULO[p]: round(downgrade[f][p], 2)
                                                   for p in orquestrados} for f in ORDEM_FATIA}),
    "pool vazio": pd.DataFrame({FATIA_ROTULO[f]: {CENARIO_ROTULO[p]: round(vazio[f][p], 2)
                                                   for p in orquestrados} for f in ORDEM_FATIA}),
}, axis=1))

## Custo de rotação: rekey de Child SA vs. reautenticação

Substitui o gráfico antigo de `keygen_pqc_end`, evento que **deixou de
existir** quando o ML-KEM local saiu do KMS — a chave agora vem da QKDN,
então não há geração local para cronometrar.

A distinção que importa: rekey de Child SA é make-before-break por
mandato da RFC 7296 e não derruba tráfego; renovar o PPK (RFC 8784) entra
na derivação no IKE_AUTH e portanto **exige reautenticação**, que é a
operação cara. Um projeto que renova PPK com frequência paga reauth na
mesma frequência.

In [ ]:
ESTAGIOS = ["child_rekey", "reauth", "ppk_fetch", "ppk_install"]
ESTAGIO_COR = {"child_rekey": "#2a78d6", "reauth": "#eb6834",
               "ppk_fetch": "#1baf7a", "ppk_install": "#eda100"}
ESTAGIO_ROTULO = {"child_rekey": "rekey Child SA", "reauth": "reautenticação",
                  "ppk_fetch": "busca de PPK", "ppk_install": "instalação de PPK"}

contagem = {e: {p: float(np.mean([len(r["duracoes"].get(e, [])) for r in execucoes[p]]))
                for p in orquestrados} for e in ESTAGIOS}

fig, axes = plt.subplots(1, 2, figsize=(12, 4.6))
barras(axes[0], orquestrados, ESTAGIOS, contagem, ESTAGIO_COR, ESTAGIO_ROTULO)
axes[0].set_ylabel("operações por execução (média)")
axes[0].set_title("Quantas vezes cada estágio roda", loc="left", pad=10)
axes[0].legend(ncol=2, loc="upper left")

# duracao: distribuicao, nao media — a cauda e o que ameaca a URLLC
amostras = {e: np.concatenate([np.asarray(r["duracoes"].get(e, []), dtype=float)
                               for p in orquestrados for r in execucoes[p]]
                              + [np.empty(0)]) for e in ESTAGIOS}
usaveis = [e for e in ESTAGIOS if amostras[e].size]
cx = axes[1].boxplot([amostras[e] for e in usaveis], tick_labels=[ESTAGIO_ROTULO[e] for e in usaveis],
                     showfliers=False, patch_artist=True, widths=0.55)
for caixa, e in zip(cx["boxes"], usaveis):
    caixa.set(facecolor=ESTAGIO_COR[e], edgecolor=SURFACE, linewidth=2)
for k in ("medians", "whiskers", "caps"):
    for art in cx[k]:
        art.set(color=INK2, linewidth=1.4)
axes[1].set_yscale("log")
axes[1].set_ylabel("duração (ms, escala log)")
axes[1].set_title("Duração por estágio — todas as execuções", loc="left", pad=10)
axes[1].tick_params(axis="x", rotation=18)
plt.tight_layout()
plt.show()

display(pd.DataFrame([{
    "estagio": ESTAGIO_ROTULO[e], "n": amostras[e].size,
    "p50 (ms)": round(float(np.percentile(amostras[e], 50)), 3),
    "p95 (ms)": round(float(np.percentile(amostras[e], 95)), 3),
    "p99 (ms)": round(float(np.percentile(amostras[e], 99)), 3),
} for e in usaveis]).set_index("estagio"))

## Isolamento entre fatias

Dois números por fatia: pacotes **classificados** pelo `iptables mangle`
e pacotes efetivamente **cifrados** pela SA daquela fatia. A grandeza de
interesse é a **lacuna** entre os dois, que mede vazamento de
classificação — e ela é da ordem de 1%, então plotar as contagens cruas
lado a lado não mostra nada: as duas barras ficam idênticas, e a eMBB a
13 mil pacotes esmaga a mIoT a 280. O gráfico é da lacuna; as contagens
ficam na tabela.

Lacuna negativa não é erro: a SA pode cifrar um pacote a mais que o
contador do `mangle` registrou no instante da amostra, porque os dois
contadores são lidos em momentos diferentes.

`conflitos` conta SPIs que apareceram com mais de uma marca — tem de ser
zero, senão duas fatias compartilham SA e o isolamento não existe.

In [ ]:
iso = []
for p in presentes:
    for mark, fatia in MARK_POR_FATIA.items():
        c = [r["classificados"].get(mark, 0) for r in execucoes[p]]
        f = [r["cifrados"].get(mark, 0) for r in execucoes[p]]
        if not any(c) and not any(f):
            continue
        clas, cif = float(np.median(c)), float(np.median(f))
        iso.append({"cenario": p, "fatia": fatia, "classificados": clas,
                    "cifrados": cif,
                    "lacuna_pct": (clas - cif) / clas * 100 if clas else np.nan})
iso = pd.DataFrame(iso)
conflitos = sum(r["conflitos"] for p in presentes for r in execucoes[p])
TOLERANCIA_PCT = 3.0

if not iso.empty:
    cen_iso = [p for p in presentes if p in set(iso["cenario"])]
    lacuna = {f: {p: float(iso[(iso["cenario"] == p) & (iso["fatia"] == f)]
                            ["lacuna_pct"].iloc[0])
                  if not iso[(iso["cenario"] == p) & (iso["fatia"] == f)].empty
                  else np.nan for p in cen_iso} for f in ORDEM_FATIA}

    fig, ax = plt.subplots(figsize=(9, 4.6))
    barras(ax, cen_iso, ORDEM_FATIA, lacuna, FATIA_COR, FATIA_ROTULO)
    ax.axhline(TOLERANCIA_PCT, color=INK2, lw=1.2, ls="--")
    ax.annotate(f"tolerância ({TOLERANCIA_PCT:.0f}%)", (0.01, TOLERANCIA_PCT),
                xycoords=("axes fraction", "data"), va="bottom",
                fontsize=9, color=INK2)
    ax.axhline(0, color=GRID, lw=1)
    ax.set_ylim(top=max(TOLERANCIA_PCT * 1.25,
                        float(np.nanmax(iso["lacuna_pct"])) * 1.3))
    ax.set_ylabel("lacuna classificado → cifrado (%)")
    ax.set_title("Vazamento de classificação por fatia", loc="left", pad=12)
    ax.legend(ncol=3, loc="lower right", bbox_to_anchor=(1, 1.02))
    plt.tight_layout()
    plt.show()

    t = iso.copy()
    t["lacuna %"] = t["lacuna_pct"].round(2)
    t["cenario"] = t["cenario"].map(CENARIO_ROTULO)
    display(t.drop(columns=["lacuna_pct"]).set_index(["cenario", "fatia"]))

print(f"SPIs com mais de uma marca (tem de ser 0): {conflitos}")

## Efeito da política de escalonamento

`--policy` troca a política declarada do cenário sem criar cenário novo,
então um mesmo prefixo pode conter execuções com políticas diferentes.
Só faz sentido comparar dentro do cenário, onde o resto é igual.

In [ ]:
pol = defaultdict(lambda: defaultdict(list))
for p in orquestrados:
    for r in execucoes[p]:
        if r["politica"]:
            pol[p][r["politica"]].append(float(np.percentile(r["latencias"], 50)))

comparaveis = {p: d for p, d in pol.items() if len(d) > 1}
if not comparaveis:
    print("nenhum cenario com mais de uma politica na janela — nada a comparar")
    print({CENARIO_ROTULO[p]: {k: len(v) for k, v in d.items()} for p, d in pol.items()})
else:
    nomes = sorted({k for d in comparaveis.values() for k in d})
    cores = dict(zip(nomes, ["#2a78d6", "#eb6834", "#1baf7a"]))
    cen = [p for p in orquestrados if p in comparaveis]
    valores = {k: {p: float(np.median(comparaveis[p].get(k) or [np.nan])) for p in cen}
               for k in nomes}
    fig, ax = plt.subplots(figsize=(8, 4.4))
    barras(ax, cen, nomes, valores, cores, {k: k for k in nomes})
    ax.set_ylabel("latência URLLC p50 (ms, mediana entre execuções)")
    ax.set_title("Política de escalonamento, dentro do mesmo cenário", loc="left", pad=12)
    ax.legend(ncol=len(nomes), loc="upper left")
    plt.tight_layout()
    plt.show()
    display(pd.DataFrame({k: {CENARIO_ROTULO[p]: round(valores[k][p], 2) for p in cen}
                          for k in nomes}))

## Vazão da eMBB e perda da mIoT

A eMBB serve também de sentinela: vazão abaixo de 0,40 Mbps indica rádio
travado, e é esse o critério `radio_degradado` que descarta a execução no
carregador. O que sobrou aqui já passou por esse filtro.

In [ ]:
embb = {p: [v for r in execucoes[p] for v in r["embb"]] for p in presentes}
miot = {p: [v for r in execucoes[p] for v in r["miot"]] for p in presentes}

fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.4))
for ax, dados, titulo, eixo in (
        (axes[0], embb, "Vazão da eMBB", "Mbps"),
        (axes[1], miot, "Perda da mIoT", "% de pacotes perdidos")):
    cen = [p for p in presentes if dados[p]]
    cx = ax.boxplot([dados[p] for p in cen], showfliers=False, patch_artist=True,
                    widths=0.55,
                    tick_labels=[CENARIO_ROTULO[p] for p in cen])
    for caixa, p in zip(cx["boxes"], cen):
        caixa.set(facecolor=CENARIO_COR[p], edgecolor=SURFACE, linewidth=2)
    for k in ("medians", "whiskers", "caps"):
        for art in cx[k]:
            art.set(color=INK2, linewidth=1.4)
    ax.set_ylabel(eixo)
    ax.set_title(titulo, loc="left", pad=10)
    ax.tick_params(axis="x", rotation=25)
axes[0].axhline(0.40, color=INK2, lw=1.2, ls="--")
axes[0].annotate("piso de rádio saudável (0,40)", (0.02, 0.40), xycoords=("axes fraction", "data"),
                 va="bottom", fontsize=9, color=INK2)
plt.tight_layout()
plt.show()

display(pd.DataFrame([{"cenario": CENARIO_ROTULO[p],
                       "eMBB Mbps p50": round(float(np.median(embb[p])), 3) if embb[p] else None,
                       "mIoT perda % p50": round(float(np.median(miot[p])), 3) if miot[p] else None}
                      for p in presentes]).set_index("cenario"))